# Presentation
## Settings

In [ ]:
KERNEL = "local"  # "local" | "kaggle"

In [ ]:
from IPython.core.magic import register_cell_magic


@register_cell_magic
def kernel(line, cell):
    """Run this cell only when KERNEL matches the given argument, e.g. `%%kernel kaggle`."""
    if line.strip() == KERNEL:
        get_ipython().run_cell(cell)

In [ ]:
%%kernel local
%cd ../..

## Prezentarea problemei

In [ ]:
import pandas as pd

from mts.helpers.imc.metric import read_csv, tth_from_csv

submission_filepath = "iterations/0138/submission.csv"
train_labels_filepath = "data/image-matching-challenge-2025/train_labels.csv"

submission_samples = read_csv(submission_filepath)
train_samples = read_csv(train_labels_filepath)

In [ ]:
submission_samples["ETs"]

In [ ]:
from dataclasses import dataclass

from app.imc2025.prediction import Prediction
from mts.core.types import Rigid3dDict


@dataclass
class GTSample:
    sample: Prediction
    gt_pose: Rigid3dDict
    scene_name: str


In [ ]:
from app.imc2025.prediction import load_from_csv

samples = load_from_csv("data/image-matching-challenge-2025", "train_labels.csv")

In [ ]:
type(samples)

In [ ]:
samples_map = {sample.filename: sample for sample in samples[0]["ETs"]}

In [ ]:
gt_samples = []
for scene_name, scene_poses in train_samples["ETs"].items():
    for image_filename, pose_dict in scene_poses.items():
        gt_samples.append(
            GTSample(
                samples_map[image_filename],
                pose_dict,
                scene_name,
            )
        )

from mts.utils.iterate import group_by

per_scene_gt_samples = group_by(gt_samples, key=lambda x: x.scene_name)

In [ ]:
et_gt_samples = per_scene_gt_samples["ET"]

In [ ]:
from app.imc2025.eval.summary import compute_eval_summary, SampleMap, BestEvalAlignment

In [ ]:
thresholds_map, th_n = tth_from_csv(
    "data/image-matching-challenge-2025/train_thresholds.csv"
)

In [ ]:
datasets_names = [
    "imc2023_haiper",
    "ETs",
    "imc2023_theather_imc2024_church",
    # "amy_gardens",
    # "pt_piazzasanmarco_grandplace",
    # "fbk_vineyard",
    # "pt_brandenburg_british_buckingham",
    # "imc2023_heritage",
    # "pt_sacrecoeur_trevi_tajmahal",
    # "pt_stpeters_stpauls",
    "imc2024_dioscuri_baalshamin",
    # "imc2024_lizard_pond",
    # "stairs",
]

In [ ]:
dataset_name = "ETs"

In [ ]:
eval_map = compute_eval_summary(
    train_samples, submission_samples, thresholds_map, datasets_names
)

In [ ]:
idx = 1

In [ ]:
dataset_eval = eval_map["ETs"]
best_alignment = dataset_eval["best_alignment"]

In [ ]:
dataset_eval.keys()

In [ ]:
matched_idx = 1
gt_cluster_name = dataset_eval["best_alignment"]["gt_scenes"][matched_idx]
user_cluster_name =dataset_eval["best_alignment"]["user_scenes"][matched_idx]

In [ ]:
from app.imc2025.eval.summary import align_poses


aligned_poses = align_poses("ETs", idx, submission_samples, best_alignment)

In [ ]:
from mts.viz.plotly.figure import init_figure
from mts.viz.plotly.rigid3d import plot_pose_dicts
from mts.viz.plotly.geometry import Rectangle

In [ ]:
submission_samples["ETs"].keys()

## Reconstrucția 3D bazată pe mișcare - <i>Structure from Motion (SfM)</i>

<div style="display:flex; align-items:center; justify-content:center; gap:16px; font-family:sans-serif;">

  <div style="text-align:center;">
    <div style="margin-bottom:4px;">Set de imagini</div>
    <div style="border:2px solid red; background:#ffe5e5; padding:10px; display:grid; grid-template-columns:repeat(2, 80px); gap:10px;">
      <figure style="margin:0; text-align:center;"><figcaption><i>Imagine<sub>1</sub></i></figcaption><img src="assets/fountain_image_056.jpg" width="80" height="120"></figure>
      <figure style="margin:0; text-align:center;"><figcaption><i>Imagine<sub>2</sub></i></figcaption><img src="assets/fountain_image_071.jpg" width="80" height="120"></figure>
      <figure style="margin:0; text-align:center;"><figcaption><i>Imagine<sub>3</sub></i></figcaption><img src="assets/fountain_image_082.jpg" width="80" height="120"></figure>
      <figure style="margin:0; text-align:center;"><figcaption><i>Imagine<sub>4</sub></i></figcaption><img src="assets/fountain_image_012.jpg" width="80" height="120"></figure>
    </div>
  </div>

  <div style="font-size:32px; font-weight:bold;">&#10132;</div>

  <div style="text-align:center;">
    <div style="border:2px solid blue; background:#b3b3ff; border-radius:8px; padding:10px; width:150px; text-align:center; font-size:14px;">Sistem de reconstrucție - (<i>Structure from Motion (SfM)</i>)</div>
  </div>

  <div style="font-size:32px; font-weight:bold;">&#10132;</div>

  <div style="text-align:center;">
    <div style="margin-bottom:4px;">Reconstrucție</div>
    <div style="border:2px solid green; background:#e5ffe5; padding:6px;">
      <img src="assets/reconstruction.gif" width="190" height="320" style="border-radius:15px; object-fit:cover;">
    </div>
  </div>

</div>

In [ ]:
from mts.viz.plotly.figure import init_figure


fig = init_figure()
plot_pose_dicts(
    train_samples[dataset_name][best_alignment["gt_scenes"][idx]].values(),
    names=["gt"],
    colors=["green"],
    fig=fig,
)

plot_pose_dicts(
    aligned_poses,
    names=["new"],
    colors=["red"],
    fig=fig,
)

## Provocări

**Scene multiple**

<div style="display:flex; justify-content:center;">
  <div style="display:inline-block;">
    <div style="border:2px solid red; background:#ffe5e5; border-radius:15px; padding:8px 12px;">
    <div style="display:flex; align-items:center; gap:8px;"><div style="width:110px; text-align:right; font-size:12px; color:#555;">scena <i>bike</i> &#10140;</div><div><img src="assets/slides/ch1/mutiple-scene/bike_image_004.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/mutiple-scene/bike_image_049.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/mutiple-scene/bike_image_088.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"></div></div>
    <div style="display:flex; align-items:center; gap:8px;"><div style="width:110px; text-align:right; font-size:12px; color:#555;">scena <i>chairs</i> &#10140;</div><div><img src="assets/slides/ch1/mutiple-scene/chairs_image_103.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/mutiple-scene/chairs_image_141.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/mutiple-scene/chairs_image_073.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"></div></div>
    <div style="display:flex; align-items:center; gap:8px;"><div style="width:110px; text-align:right; font-size:12px; color:#555;">scena <i>fountain</i> &#10140;</div><div><img src="assets/slides/ch1/mutiple-scene/fountain_image_033.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/mutiple-scene/fountain_image_108.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/mutiple-scene/fountain_image_143.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"></div></div>
    </div>
  </div>
</div>

**Texturi minimaliste**

<div style="display:flex; justify-content:center;">
  <div style="display:inline-block;">
    <div style="border:2px solid red; background:#ffe5e5; border-radius:15px; padding:8px 12px;">
    <div style="display:flex; align-items:center; gap:8px;"><div style="width:110px; text-align:right; font-size:12px; color:#555;">scena <i>stairs</i> &#10140;</div><div><img src="assets/slides/ch1/textura-minimalista/stairs_split_1_1710453606287.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/textura-minimalista/stairs_split_1_1710453616892.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/textura-minimalista/stairs_split_1_1710453663515.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"></div></div>
    <div style="display:flex; align-items:center; gap:8px;"><div style="width:110px; text-align:right; font-size:12px; color:#555;"> </div><div><img src="assets/slides/ch1/textura-minimalista/stairs_split_1_1710453678922.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/textura-minimalista/stairs_split_1_1710453704934.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/textura-minimalista/stairs_split_2_1710453736752.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"></div></div>
    </div>
  </div>
</div>

**Scală variantă**

<div style="display:flex; justify-content:center;">
  <div style="display:inline-block;">
    <div style="border:2px solid red; background:#ffe5e5; border-radius:15px; padding:8px 12px;">
    <div style="display:flex; align-items:center; gap:8px;"><div style="width:110px; text-align:right; font-size:12px; color:#555;">centimetri &#10140;</div><div><img src="assets/slides/ch1/scala/another_et_another_et007.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/scala/et_et003.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/scala/et_et001.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"></div></div>
    <div style="display:flex; align-items:center; gap:8px;"><div style="width:110px; text-align:right; font-size:12px; color:#555;">metri &#10140;</div><div><img src="assets/slides/ch1/scala/british_museum_28309287_5482270912.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/scala/brandenburg_gate_76177997_2293073399.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/scala/brandenburg_gate_52599802_11381897224.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"></div></div>
    </div>
  </div>
</div>

**Scene similare**

<div style="display:flex; justify-content:center;">
  <div style="display:inline-block;">
    <div style="border:2px solid red; background:#ffe5e5; border-radius:15px; padding:8px 12px;">
    <div style="display:flex; align-items:center; gap:8px;"><div style="width:110px; text-align:right; font-size:12px; color:#555;">scena <i>stairs</i>1 &#10140;</div><div><img src="assets/slides/ch1/scene-similare/stairs_split_1_1710453576271.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/scene-similare/stairs_split_1_1710453626698.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/scene-similare/stairs_split_1_1710453667117.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"></div></div>
    <div style="display:flex; align-items:center; gap:8px;"><div style="width:110px; text-align:right; font-size:12px; color:#555;">scena <i>stairs</i>2 &#10140;</div><div><img src="assets/slides/ch1/scene-similare/stairs_split_2_1710453728949.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/scene-similare/stairs_split_2_1710453765165.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"><img src="assets/slides/ch1/scene-similare/stairs_split_2_1710453798181.jpg" style="width:110px; height:74px; object-fit:cover; margin:3px;"></div></div>
    </div>
  </div>
</div>

# Pipeline

<div style="display:flex; align-items:center; justify-content:center; gap:16px; font-family:sans-serif;">

  <div style="text-align:center;">
    <div style="margin-bottom:4px;">Set de imagini</div>
    <div style="border:2px solid red; background:#ffe5e5; padding:10px; display:grid; grid-template-columns:repeat(2, 80px); gap:10px;">
      <figure style="margin:0; text-align:center;"><figcaption><i>Imagine<sub>1</sub></i></figcaption><img src="assets/fountain_image_056.jpg" width="80" height="120"></figure>
      <figure style="margin:0; text-align:center;"><figcaption><i>Imagine<sub>2</sub></i></figcaption><img src="assets/fountain_image_071.jpg" width="80" height="120"></figure>
      <figure style="margin:0; text-align:center;"><figcaption><i>Imagine<sub>3</sub></i></figcaption><img src="assets/fountain_image_082.jpg" width="80" height="120"></figure>
      <figure style="margin:0; text-align:center;"><figcaption><i>Imagine<sub>4</sub></i></figcaption><img src="assets/fountain_image_012.jpg" width="80" height="120"></figure>
    </div>
  </div>

  <div style="font-size:32px; font-weight:bold;">&#10132;</div>

  <div style="text-align:center;">
    <div style="margin-bottom:4px;">Pipeline</div>
    <div style="border:2px solid blue; background:#e5e5ff; padding:10px; display:flex; flex-direction:column; align-items:center; gap:4px;">
      <div style="border:2px solid blue; background:white; border-radius:6px; padding:6px; width:150px; text-align:center; font-size:13px;">Propunerea perechilor</div><div style="font-size:18px; color:red;">&#8595;</div><div style="border:2px solid blue; background:#b3ffb3; border-radius:6px; padding:6px; width:150px; text-align:center; font-size:13px;">Detecția punctelor și potrivirea acestora</div><div style="font-size:18px; color:red;">&#8595;</div><div style="border:2px solid blue; background:#b3ffb3; border-radius:6px; padding:6px; width:150px; text-align:center; font-size:13px;">Crearea grafului de scenă</div><div style="font-size:18px; color:red;">&#8595;</div><div style="border:2px solid blue; background:white; border-radius:6px; padding:6px; width:150px; text-align:center; font-size:13px;">Reconstrucția incrementală</div>
    </div>
  </div>

  <div style="font-size:32px; font-weight:bold;">&#10132;</div>

  <div style="text-align:center;">
    <div style="margin-bottom:4px;">Reconstrucție</div>
    <div style="border:2px solid green; background:#e5ffe5; padding:6px;">
      <img src="assets/reconstruction.gif" width="190" height="320" style="border-radius:15px; object-fit:cover;">
    </div>
  </div>

</div>

## Propunerea perechilor

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 30%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;"></div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Extragerea imaginilor similare folosind o imagine de <i>query</i></li>
      <li>Se extrage pe baza unei reprezentări</li>
      <li>Pot apărea <i>false positive</i></li>
      <li>Rezultatul este o listă de perechi de imagini</li>
      <li>Căutarea poate fi exhaustivă sau poate folosi abordări precum: <i>approximate nearest neighbors (aNN)</i></li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <div style="display:flex; align-items:center; justify-content:center; gap:20px;"><div style="text-align:center;"><img src="assets/slides/ch2/pair-proposal/fountain_image_012.jpg" style="width:130px; border:2px solid black;"><div style="font-size:11px;">Imagine folosită<br>pentru căutare - <i>query</i></div></div><div style="display:flex; flex-direction:column;"><img src="assets/slides/ch2/pair-proposal/stairs_split_1_1710453616892.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #cc0000; margin:4px;"><img src="assets/slides/ch2/pair-proposal/fountain_image_025.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"><img src="assets/slides/ch2/pair-proposal/fountain_image_101.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"><img src="assets/slides/ch2/pair-proposal/stairs_split_1_1710453620694.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #cc0000; margin:4px;"></div><div style="display:flex; flex-direction:column;"><img src="assets/slides/ch2/pair-proposal/vineyard_split_1_frame_0910.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"><img src="assets/slides/ch2/pair-proposal/stairs_split_1_1710453659313.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #cc0000; margin:4px;"><img src="assets/slides/ch2/pair-proposal/stairs_split_1_1710453668718.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #cc0000; margin:4px;"><img src="assets/slides/ch2/pair-proposal/fountain_image_082.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"></div></div>
    <div style="text-align:center; font-style:italic; margin-top:4px;">Căutarea imaginilor similare într-un set de date.</div>
  </div>
</div>
</div>

## Propunerea perechilor

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 30%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;"></div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Extragerea imaginilor similare folosind o imagine de <i>query</i></li>
      <li>Se extrage pe baza unei reprezentări</li>
      <li>Pot apărea <i>false positive</i></li>
      <li>Rezultatul este o listă de perechi de imagini</li>
      <li>Căutarea poate fi exhaustivă sau poate folosi abordări precum: <i>approximate nearest neighbors (aNN)</i></li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <div style="display:flex; align-items:center; justify-content:center; gap:20px;"><div style="text-align:center;"><img src="assets/slides/ch2/pair-proposal/fountain_image_012.jpg" style="width:130px; border:2px solid black;"><div style="font-size:11px;">Imagine folosită<br>pentru căutare - <i>query</i></div></div><div style="display:flex; flex-direction:column;"><img src="assets/slides/ch2/pair-proposal/fountain_image_025.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"><img src="assets/slides/ch2/pair-proposal/vineyard_split_1_frame_0910.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"></div><div style="display:flex; flex-direction:column;"><img src="assets/slides/ch2/pair-proposal/fountain_image_101.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"><img src="assets/slides/ch2/pair-proposal/fountain_image_082.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"></div></div>
    <div style="text-align:center; font-style:italic; margin-top:4px;">Imaginile similare returnate.</div>
  </div>
</div>
</div>

## Filtrarea perechilor

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 35%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Pași</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Un pas adițional de ordonare</li>
      <li>Se poate realiza folosind verificarea geometrică</li>
      <li>Este nevoie de puncte, descriptori și potriviri</li>
      <li>Abordare similară cu sistemele de <i>retrieval</i></li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <div style="display:flex; align-items:center; justify-content:center; gap:20px;"><div style="text-align:center;"><img src="assets/slides/ch2/pair-proposal/fountain_image_012.jpg" style="width:130px; border:2px solid black;"><div style="font-size:11px;">Imagine folosită<br>pentru căutare - <i>query</i></div></div><div style="display:flex; flex-direction:column;"><img src="assets/slides/ch2/pair-proposal/fountain_image_025.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"><img src="assets/slides/ch2/pair-proposal/vineyard_split_1_frame_0910.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #cc0000; margin:4px;"></div><div style="display:flex; flex-direction:column;"><img src="assets/slides/ch2/pair-proposal/fountain_image_101.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #2e8b2e; margin:4px;"><img src="assets/slides/ch2/pair-proposal/fountain_image_082.jpg" style="width:120px; height:80px; object-fit:cover; border:3px solid #cc0000; margin:4px;"></div></div>
    <div style="text-align:center; font-style:italic; margin-top:4px;">Filtrarea imaginilor invalidate geometric.</div>
  </div>
</div>
</div>

## Potrivirea punctelor între două imagini cu texturi complexe

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 35%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Detecție și potrivire rară</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>2 pași: 1 pas de detecție și 1 pas de potrivire</li>
      <li>Detecția este aplicată per imagine</li>
      <li>Potrivirea este aplicată per pereche</li>
    </ul>
  </div>
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Detecție și potrivire densă</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Detecția și potrivirea sunt executate într-un singur pas</li>
      <li>Densitatea este mai mare</li>
      <li>Modelul este aplicat per pereche</li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch2/detection-and-matching/haiper-sparse.jpg" style="width:60%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Detecție și potrivire rară.</div>
  </div>
  <div style="text-align:center;">
    <img src="assets/slides/ch2/detection-and-matching/haiper-dense.jpg" style="width:60%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Detecție și potrivire densă.</div>
  </div>
</div>
</div>

## Potrivirea punctelor între două imagini cu texturi minimaliste și repetitive

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 35%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Detecție și potrivire rară</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Este aplicat în 2 pași</li>
      <li>Detecția este aplicată per imagine</li>
      <li>Potrivirea este aplicată per pereche</li>
    </ul>
  </div>
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Detecție și potrivire densă</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Detecția și potrivirea sunt executate într-un singur pas</li>
      <li>Densitatea este mai mare</li>
      <li>Modelul este aplicat per pereche</li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch2/detection-and-matching/stairs-sparse.jpg" style="width:85%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Detecție și potrivire rară.</div>
  </div>
  <div style="text-align:center;">
    <img src="assets/slides/ch2/detection-and-matching/stairs-dense.jpg" style="width:85%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Detecție și potrivire densă.</div>
  </div>
</div>
</div>

## Construirea grafului de scenă

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 30%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Construcția grafului de scenă</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Se folosesc perechile propuse în pasul de propunere</li>
      <li>Se calculează potrivirile între două imagini</li>
      <li>Verificare geometrică</li>
      <li>Filtrare folosind un prag minim de potriviri</li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch2/scene_graph_1.svg" style="width:100%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Ștergerea perechilor nevalidate.</div>
  </div>
</div>
</div>

## Construcția grafului de scenă

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 30%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Construcția grafului de scenă</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Se folosesc perechile propuse în pasul de propunere</li>
      <li>Se calculează potrivirile între două imagini</li>
      <li>Verificare geometrică</li>
      <li>Filtrare folosind un prag minim de potriviri</li>
      <li>Perechile nevalidate sunt șterse</li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch2/scene_graph_2.svg" style="width:100%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Graful de scenă.</div>
  </div>
</div>
</div>

## Reconstrucția incrementală și optimizarea globală

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 30%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Reconstrucția incrementală și optimizare globală</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Folosește graful scenei</li>
      <li>Inițializată cu o pereche de imagini</li>
      <li>Imaginile se adaugă pe rând</li>
      <li>Realizată folosind sistemul <i>COLMAP</i></li>
      <li>Este nevoie de puncte, descriptori și potriviri</li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch2/incremental_ba.svg" style="width:100%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Reconstrucție incrementală și optimizarea globală.</div>
  </div>
</div>
</div>

## Construcția grafului de scenă

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 30%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Separarea scenelor</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Realizată folosind <i>COLMAP</i></li>
      <li>Realizată folosind un algoritm de detecție a comunităților dintr-un graf</li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch2/scene_separation.svg" style="width:100%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Separarea scenelor în două clustere.</div>
  </div>
</div>
</div>

## <i>Pipeline</i> de reconstrucție

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 30%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Pipeline de reconstrucție</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Pipeline generic, ușor de schimbat o componentă</li>
      <li>Experimentare cu mai multe modele de reprezentare globală a imaginii - <i>image embedding</i></li>
      <li>Modelul <i>mast3r</i> folosit pentru potrivire densă</li>
      <li>Pasul de tranzitivitate este folosit pentru reducerea timpului de procesare</li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch3/pipeline.svg" style="width:100%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;"><i>Pipeline</i> de reconstrucție incrementală.</div>
  </div>
</div>
</div>

## Probleme

### Viteza de procesare
* Potrivirea densa este aplicata per pereche de imagini. 
* Numarul de perechi creste cuadratic
* Modelele de potrivire nu sunt proiectate pt a fi folosite in SfM standard

#### Restructurarea modelului sa fie folosit pt sfm 

**Arhitectura unui model de detecție și potrivire densă**

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:0 0 30%;">
  <div style="border:1px solid #ddd; border-radius:4px; overflow:hidden; margin-bottom:12px;">
    <div style="background:#a30100; color:white; padding:4px 8px; font-weight:bold;">Arhitectura unui model de detecție și potrivire densă</div>
    <ul style="margin:6px 0; padding:4px 8px 4px 28px; background:#f7f7f7;">
      <li>Compus din 3 nivele: codificare, decodificare, și agregare</li>
      <li>Modulul de decodificare refolosește parametrii</li>
      <li>Modulul de decodificare reprezentat printr-un <i>cross-attention</i></li>
    </ul>
  </div>
</div>
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch3/dense_arch.svg" style="width:100%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Arhitectura unui model de detecție și potrivire densă.</div>
  </div>
</div>
</div>

**Arhitectura unui sistem de reconstrucție folosind un model de potrivire densă**

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch3/dense_sfm_arch.svg" style="width:75%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Arhitectura <i>SfM</i> folosind un model dens.</div>
  </div>
</div>
</div>

## Tranzitivitatea

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch3/transitivity_grid.svg" style="width:80%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Tranzitivitatea pe 3 imagini.</div>
  </div>
</div>
</div>

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch3/transitivity/three-way-transivity.jpg" style="width:80%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Potrivirile tranzitive</div>
  </div>
</div>
</div>

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/slides/ch3/transitivity/minmal-three-way-transivity.jpg" style="width:60%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;">Potrivirile tranzitive.</div>
  </div>
</div>
</div>

### Clasament
* 13/30
* scor 38.96
* cel mai bun scor 65.55

### Posibile Imbunatatiri

<div style="display:flex; gap:16px; align-items:flex-start;">
<div style="flex:1;">
  <div style="text-align:center;">
    <img src="assets/refinement.png" style="width:100%; ">
    <div style="text-align:center; font-style:italic; margin-top:4px;"></div>
  </div>
</div>
</div>

## Reconstructii

In [ ]:
from pathlib import Path

import pycolmap
from hloc.utils import viz_3d

In [ ]:
reconstruction = pycolmap.Reconstruction("iterations/0426/pt_brandenburg_british_buckingham/0")
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, reconstruction)
fig

In [ ]:
reconstruction = pycolmap.Reconstruction("iterations/0426/pt_brandenburg_british_buckingham/1")
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, reconstruction)
fig

In [ ]:
reconstruction = pycolmap.Reconstruction("iterations/0426/pt_brandenburg_british_buckingham/2")
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, reconstruction)
fig

In [ ]:
reconstruction = pycolmap.Reconstruction("iterations/0426/imc2023_haiper/0")
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, reconstruction)
fig

In [ ]:
reconstruction = pycolmap.Reconstruction("iterations/0426/imc2023_haiper/1")
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, reconstruction)
fig

In [ ]:
reconstruction = pycolmap.Reconstruction("iterations/0426/ETs/1")
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, reconstruction)
fig

In [ ]:
reconstruction = pycolmap.Reconstruction("iterations/0426/ETs/0")
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, reconstruction)
fig